In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path(".")

files = [
    ("A_7_5g",  ROOT/"ACoG_7_5g"/"ACoG_7_5g_clean.csv"),
    ("A_15g",   ROOT/"ACoG_15g"/"ACoG_15g_clean.csv"),
    ("A_18g",   ROOT/"ACoG_18g"/"ACoG_18g_clean.csv"),
    ("A_22_5g", ROOT/"ACoG_22_5g"/"ACoG_22_5g_clean.csv"),
]

def load(path):
    df = pd.read_csv(path)
    # Arreglar por si quedaron nombres raros:
    df = df.rename(columns={c.lower().strip():c for c in df.columns})
    if not {"nm","A"}.issubset(df.columns):
        # Si venía como x,y entonces lo corregimos
        c0, c1 = df.columns[:2]
        df = df.rename(columns={c0:"nm", c1:"A"})
    df["nm"] = pd.to_numeric(df["nm"], errors="coerce")
    df["A"]  = pd.to_numeric(df["A"], errors="coerce")
    return df.dropna()[["nm","A"]].sort_values("nm").drop_duplicates()

# ---- MATRIZ SIN INTERPOLAR (INTERSECCIÓN EXACTA) ----

mat = load(files[0][1]).rename(columns={"A": files[0][0]})

for label, path in files[1:]:
    df = load(path).rename(columns={"A": label})
    mat = mat.merge(df, on="nm", how="inner")   # SOLO nm que existen en todos

mat = mat.sort_values("nm").reset_index(drop=True)

out = ROOT / "ACoG_matrix.csv"
mat.to_csv(out, index=False)

print("✔ Matriz generada correctamente:")
print(out)
display(mat.head())


✔ Matriz generada correctamente:
ACoG_matrix.csv


,nm,A_7_5g,A_15g,A_18g,A_22_5g
0,231.0,1.486,1.620,1.587,2.386
1,231.5,1.468,1.600,1.567,2.340
2,232.0,1.451,1.580,1.547,2.299
3,232.5,1.434,1.561,1.528,2.260
4,233.0,1.417,1.542,1.509,2.224
